# Obligatorio - Taller de Deep Learning

**Fecha de entrega:** 8/12/2026 hasta las 21:00 (hora de Montevideo). Es también el cierre de la competencia en Kaggle (último envío).
**Puntaje máximo:** 50 puntos

**Integrantes:** [Nombre(s) y Apellido(s)]

## Obligatorio

El objetivo de este obligatorio es evaluar su conocimiento en Deep Learning mediante la implementación completa de un modelo de segmentación de imágenes basado en el paper [**"U-Net: Convolutional Networks for Biomedical Image Segmentation"**](https://arxiv.org/pdf/1505.04597). Toda la implementación debe realizarse desde cero utilizando PyTorch, y los estudiantes tendrán la libertad de ajustar ciertos hiperparámetros y configuraciones mientras mantengan la esencia del paper original.

La tarea de este año es la **segmentación de lesiones mamarias en imágenes de ecografía**: para cada imagen, el modelo debe indicar qué píxeles pertenecen a la lesión.

### **Competencia en Kaggle**

Además, como parte de este obligatorio, participarán en una competencia privada en Kaggle donde se les proporcionará un dataset de test oculto (sin target). Deberán subir sus predicciones a Kaggle y se evaluarán en función de la métrica **Dice Coefficient (Coeficiente de Dice)**. Esta competencia les permitirá comparar sus resultados con los de sus compañeros en un entorno real de evaluación.

Las imágenes de test **provienen de otros centros y fueron adquiridas con otros ecógrafos** que las imágenes de entrenamiento. Por lo tanto, un buen resultado en validación no garantiza un buen resultado en test: el modelo debe **generalizar a un dominio distinto** del que vio durante el entrenamiento.

### **¿Qué es el Dice Coefficient?**
El **Dice Coefficient**, también conocido como F1-score para segmentación, es una métrica utilizada para evaluar la similitud entre la predicción y la anotación manual en tareas de segmentación. Se define de la siguiente manera:

$$
\text{Dice} = \frac{2 \cdot |A \cap B|}{|A| + |B|}
$$

Donde:
- $A$ es el conjunto de píxeles predichos como pertenecientes a la clase positiva.
- $B$ es el conjunto de píxeles verdaderos pertenecientes a la clase positiva.
- $|A \cap B|$ es la intersección de $A$ y $B$, es decir, los píxeles correctamente predichos como positivos.

Un valor de Dice de **1** indica una predicción perfecta, mientras que un valor de **0** indica que no hay coincidencia entre la predicción y el valor verdadero. En la competencia, el Dice se calcula **por imagen** y luego se promedia sobre las imágenes de test. Todas las imágenes de test contienen una lesión, por lo que **la predicción de cada imagen debe tener al menos un píxel de lesión**: si alguna fila tiene la máscara vacía, Kaggle rechaza el envío completo. Para aprobar el obligatorio es necesario obtener un puntaje de al menos **0.65** en la tabla pública de Kaggle.

### **Criterios a Evaluar**

1. **Análisis del Dataset (5 puntos):**
   - Exploración y visualización del dataset para comprender su estructura y características (tamaños de imagen, proporción de píxeles de lesión, pacientes, equipos de adquisición, etc.).
   - Comparación entre los equipos de adquisición del conjunto de entrenamiento, e identificación de las diferencias que pueden afectar al modelo cuando se aplique a imágenes de otros equipos.
   - Justificación de las decisiones tomadas en el preprocesamiento de datos, como el tamaño de entrada del modelo, la normalización, el aumento de datos (data augmentation) y la partición del dataset en conjuntos de entrenamiento y validación.

2. **Implementación Correcta del Modelo U-Net (20 puntos):**
   - Construcción de la arquitectura U-Net siguiendo la estructura descrita en el paper, permitiendo ajustes como el número de filtros, funciones de activación y métodos de inicialización de pesos.
   - Se aceptan mejoras, por ejemplo batch normalization u otras funciones de activación.

3. **Entrenamiento del Modelo (10 puntos):**
   - Configuración adecuada del ciclo de entrenamiento, incluyendo la elección de la función de pérdida y del optimizador (Adam, SGD, etc.).
   - Uso de técnicas de regularización para mejorar la generalización del modelo, como el dropout, batch normalization y data augmentation.
   - Gráficas y análisis de la evolución del entrenamiento, mostrando las curvas de pérdida y métricas relevantes tanto en el conjunto de entrenamiento como en el de validación.
   - Puede utilizarse experimentación con hiperparámetros con Weights & Biases (W&B) para optimizar el rendimiento del modelo. Este punto es opcional, pero se valorará positivamente si se justifica su uso y se presentan resultados claros.

4. **Evaluación de Resultados (10 puntos):**
   - Evaluación exhaustiva del modelo utilizando métricas de segmentación como **Dice Coefficient**.
   - Análisis detallado de los resultados, incluyendo un análisis de errores para identificar y discutir casos difíciles.
   - Análisis del efecto del cambio de dominio: qué tipo de imágenes o lesiones se segmentan peor y por qué.
   - Visualización de ejemplos representativos de segmentaciones correctas e incorrectas, comparando con las etiquetas manuales proporcionadas en el dataset.

5. **Participación y Resultados en la Competencia Kaggle (5 puntos):**
   - Participación activa en la competencia de Kaggle, con al menos un (1) envío de predicciones.
   - Puntaje obtenido en la tabla de posiciones de Kaggle, evaluado en base al **Dice Coefficient** en el conjunto de test oculto. Es necesario obtener al menos **0.65** en la tabla pública para aprobar el obligatorio.

   Notas:
   - **Cualquier decisión debe ser justificada en el notebook.**
   - El **Dice Coefficient** es la métrica utilizada para evaluar el desempeño de los modelos de segmentación en esta competencia.

### **Run-Length Encoding (RLE)**

Dado que no se suben las imágenes segmentadas directamente a Kaggle, se requiere usar **Run-Length Encoding (RLE)** para comprimir las máscaras de predicción en una cadena de texto que será evaluada. El **RLE** es una técnica de compresión donde se representan secuencias consecutivas de píxeles en formato `start length`, indicando la posición de inicio y la longitud de cada secuencia de píxeles positivos.

Para calcular el **RLE**, se sigue el siguiente proceso:

1. Se aplanan las máscaras predichas en un solo vector (por columnas, orden Fortran). La máscara es un array de numpy de forma `(alto, ancho)` con valores 0 y 1.
2. Se identifican los píxeles con valor positivo (1) y se calculan las secuencias consecutivas.
3. Se registra la posición de inicio de cada secuencia (comenzando en 1) y su longitud en formato `start length`.

Este formato comprimido se sube a Kaggle en lugar de las imágenes segmentadas.

**Importante:** no se aceptan máscaras vacías (ver la sección anterior). Las imágenes de test tienen tamaños distintos. La máscara de cada imagen debe codificarse **con el tamaño original de esa imagen**. Si el modelo trabaja con otro tamaño, la predicción debe llevarse de vuelta al tamaño original antes de calcular el RLE.

#### **Ejemplo de RLE**

```python
import numpy as np

def rle_encode(mask):
    pixels = np.array(mask).flatten(order='F')  # Aplanar la máscara en orden Fortran
    pixels = np.concatenate([[0], pixels, [0]])  # Añadir ceros al principio y final
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1  # Encontrar transiciones
    runs[1::2] = runs[1::2] - runs[::2]  # Calcular longitudes
    return ' '.join(str(x) for x in runs)

mask = np.array([[0, 0, 1, 0, 0],
                 [0, 1, 1, 1, 0],
                 [1, 1, 1, 0, 0],
                 [0, 0, 0, 1, 1]])

print(rle_encode(mask))
```

> **Salida:** 3 1 6 2 9 3 14 1 16 1 20 1


### **Sobre el Dataset**

El dataset contiene imágenes de ecografía mamaria en escala de grises, cada una con una lesión (benigna o maligna) segmentada manualmente por especialistas.

- **Entrenamiento** (`train/`): imágenes (`train/images/`) y máscaras binarias (`train/masks/`, con el mismo nombre de archivo que la imagen; 255 = lesión, 0 = fondo). El archivo `train/train.csv` indica, para cada imagen, el paciente (`case`), el equipo de adquisición (`device`) y la patología (`pathology`). Un mismo paciente puede tener más de una imagen: los sufijos `-l` y `-r` del nombre corresponden a dos vistas de la misma lesión, y `-s` a una vista única.
- **Test** (`test/images/`): imágenes sin máscara, de otros centros y otros equipos. Cada imagen de test contiene al menos una lesión. **Las imágenes de test se usan únicamente para generar las predicciones que se envían a Kaggle**: no pueden usarse para entrenar, ajustar ni analizar el modelo.
- Las imágenes se entregan **en su resolución original**, que varía de una imagen a otra. La elección del tamaño de entrada del modelo es parte del trabajo.
- `sample_submission.csv` muestra el formato de la entrega: una fila por imagen de test, con las columnas `id` y `encoded_pixels` (cada fila contiene un único píxel, solo como ejemplo del formato).

### **Reglas de la Competencia**

- **Solo se pueden usar los datos de entrenamiento provistos.** No se permite usar datos externos (incluidas las versiones originales de los datasets citados en la sección *Datos y Atribución*) ni modelos o pesos pre-entrenados: la U-Net se entrena desde cero.
- Cada equipo puede realizar como máximo **dos (2) envíos por día** y debe seleccionar **dos (2)** envíos para la evaluación final.
- La tabla pública se calcula con el 80 % de las imágenes de test y es la que se usa para la aprobación. La posición final se define con el 20 % restante (tabla privada), que se revela al cierre de la competencia.
- **Premio:** el equipo en el primer puesto de la tabla privada recibe un ejemplar del libro [*The Welch Labs Illustrated Guide to AI*](https://www.welchlabs.com/ai-book), siempre que apruebe la defensa del obligatorio respondiendo correctamente al menos cuatro (4) de las cinco (5) preguntas.

### **Instrucciones de Entrega**

- Deberán entregar un Jupyter Notebook (.ipynb) que contenga todo el código y las explicaciones necesarias para ejecutar la implementación, el entrenamiento y la evaluación del modelo.
- **El notebook principal no puede superar las 50 páginas una vez exportado a PDF** (con sus salidas: tablas, gráficas e imágenes). Se corrige el notebook principal; lo que exceda ese límite no se tiene en cuenta.
- Opcionalmente, pueden entregar **notebooks secundarios** con material de apoyo. Por ejemplo, `experimentos_augmentation.ipynb` con todos los entrenamientos de una búsqueda de hiperparámetros: el notebook principal incluye solo la tabla de resultados y las conclusiones, y referencia al secundario para el detalle. Los notebooks secundarios no son obligatorios.
- El notebook debe incluir secciones bien documentadas explicando las decisiones de diseño del modelo, los experimentos realizados, y los resultados obtenidos.
- El código debe estar escrito de manera clara.
- La entrega debe realizarse a través de la plataforma de gestión de ORT (gestion.ort.edu.uy) antes de la fecha límite.

### **Materiales Adicionales**

Para facilitar su trabajo, pueden consultar los siguientes recursos:

- [U-Net: Convolutional Networks for Biomedical Image Segmentation (paper original)](https://arxiv.org/abs/1505.04597)
- [Documentación de PyTorch](https://pytorch.org/docs/stable/index.html)
- [Tutoriales y recursos adicionales en Kaggle](https://www.kaggle.com/)
- [Convolución Transpuesta](https://d2l.ai/chapter_computer-vision/transposed-conv.html)

### **Datos y Atribución**

- **Entrenamiento:** dataset BUS-BRA (ver la licencia en `train/LICENSE.txt`).

  > Gómez-Flores, W., Gregorio-Calas, M. J., & Coelho de Albuquerque Pereira, W. (2024). BUS-BRA: A breast ultrasound dataset for assessing computer-aided diagnosis systems. *Medical Physics*, 51(4), 3110–3123. https://doi.org/10.1002/mp.16812

- **Test:** dataset Breast-Lesions-USG, licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Las imágenes de test fueron modificadas y renombradas para la competencia, y se descartaron los casos sin lesión.

  > Pawłowska, A., Ćwierz-Pieńkowska, A., Domalik, A., Jaguś, D., Kasprzak, P., Matkowski, R., Fura, Ł., Nowicki, A., & Zolek, N. (2024). A Curated Benchmark Dataset for Ultrasound Based Breast Lesion Analysis (Breast-Lesions-USG) (Version 1) [dataset]. The Cancer Imaging Archive. https://doi.org/10.7937/9WKK-Q141

  > Pawłowska, A., et al. (2024). A curated benchmark dataset for ultrasound based breast lesion analysis. *Scientific Data*, 11, 148. https://doi.org/10.1038/s41597-024-02984-z

Ambos datasets son públicos. Usar sus máscaras originales (o cualquier otra anotación externa) para generar o corregir predicciones viola las reglas de la competencia.

### **Enlace a la Competencia**

La competencia abre el **miércoles 7 de octubre de 2026**. Es privada: para unirse, cada integrante debe ingresar con su cuenta de Kaggle al [enlace de invitación](https://www.kaggle.com/t/c6463c3f3c4c45918b447ebd582c5f5f). Luego, la competencia queda disponible en [kaggle.com/competitions/tdl-obligatorio-2026](https://www.kaggle.com/competitions/tdl-obligatorio-2026).

---